In [2]:
import duckdb

In [3]:
duckdb.sql("""
SELECT COUNT(*) AS linhas,
         COUNT(DISTINCT order_id) AS ids_distintos
         FROM 'data/raw/olist_orders_dataset.csv'
         """).show()

┌────────┬───────────────┐
│ linhas │ ids_distintos │
│ int64  │     int64     │
├────────┼───────────────┤
│  99441 │         99441 │
└────────┴───────────────┘



In [4]:
duckdb.sql("""
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT (order_id, order_item_id)) AS combinacoes_distintas
FROM 'data/raw/olist_order_items_dataset.csv'
""").show()

┌────────┬───────────────────────┐
│ linhas │ combinacoes_distintas │
│ int64  │         int64         │
├────────┼───────────────────────┤
│ 112650 │                112650 │
└────────┴───────────────────────┘



In [5]:
CANDIDATAS = [
    ("olist_orders_dataset", ["order_id"]),
    ("olist_order_items_dataset", ["order_id"]),
    ("olist_order_items_dataset", ["order_id", "order_item_id"]),
    ("olist_order_payments_dataset", ["order_id", "payment_sequential"]),
    ("olist_order_reviews_dataset", ["review_id"]),
    ("olist_order_reviews_dataset", ["review_id", "order_id"]),
    ("olist_customers_dataset", ["customer_id"]),
    ("olist_products_dataset", ["product_id"]),
    ("olist_sellers_dataset", ["seller_id"]),
    ("olist_geolocation_dataset", ["geolocation_zip_code_prefix"]),
    ("product_category_name_translation", ["product_category_name"]),
]

In [6]:
for tabela, colunas in CANDIDATAS:
    arquivo = f"data/raw/{tabela}.csv"
    cols = ", ".join(colunas)

    linhas = duckdb.sql(f"SELECT COUNT(*) FROM '{arquivo}'").fetchone()[0]
    distintas = duckdb.sql(
        f"SELECT COUNT(*) FROM (SELECT DISTINCT {cols} FROM '{arquivo}')"
    ).fetchone()[0]

    filtro_nulos = " OR ".join(f"{c} IS NULL" for c in colunas)
    nulos = duckdb.sql(
        f"SELECT COUNT(*) FROM '{arquivo}' WHERE {filtro_nulos}"
    ).fetchone()[0]

    if linhas == distintas and nulos == 0:
        veredito = "CHAVE VALIDA"
    elif linhas != distintas:
        veredito = "REPETE"
    else:
        veredito = "TEM NULOS"

    print(f"{tabela:36} ({cols})")
    print(f"    linhas={linhas:>8}  distintas={distintas:>8}  nulos={nulos:>5}  -> {veredito}")

olist_orders_dataset                 (order_id)
    linhas=   99441  distintas=   99441  nulos=    0  -> CHAVE VALIDA
olist_order_items_dataset            (order_id)
    linhas=  112650  distintas=   98666  nulos=    0  -> REPETE
olist_order_items_dataset            (order_id, order_item_id)
    linhas=  112650  distintas=  112650  nulos=    0  -> CHAVE VALIDA
olist_order_payments_dataset         (order_id, payment_sequential)
    linhas=  103886  distintas=  103886  nulos=    0  -> CHAVE VALIDA
olist_order_reviews_dataset          (review_id)
    linhas=   99224  distintas=   98410  nulos=    0  -> REPETE
olist_order_reviews_dataset          (review_id, order_id)
    linhas=   99224  distintas=   99224  nulos=    0  -> CHAVE VALIDA
olist_customers_dataset              (customer_id)
    linhas=   99441  distintas=   99441  nulos=    0  -> CHAVE VALIDA
olist_products_dataset               (product_id)
    linhas=   32951  distintas=   32951  nulos=    0  -> CHAVE VALIDA
olist_sellers_da

In [7]:
def csv(nome):
    return f"'data/raw/{nome}.csv'"
 

In [8]:
# (tabela filha, coluna filha, tabela pai, coluna pai)
RELACOES = [
    ("olist_orders_dataset", "customer_id", "olist_customers_dataset", "customer_id"),
    ("olist_order_items_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_items_dataset", "product_id", "olist_products_dataset", "product_id"),
    ("olist_order_items_dataset", "seller_id", "olist_sellers_dataset", "seller_id"),
    ("olist_order_payments_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_reviews_dataset", "order_id", "olist_orders_dataset", "order_id"),
    (
        "olist_products_dataset",
        "product_category_name",
        "product_category_name_translation",
        "product_category_name",
    ),
]

In [9]:
print("== Registros orfaos (filha aponta para um pai que nao existe) ==")
for filha, col_f, pai, col_p in RELACOES:
    nulos = duckdb.sql(
        f"SELECT COUNT(*) FROM {csv(filha)} WHERE {col_f} IS NULL"
    ).fetchone()[0]
    orfas = duckdb.sql(
        f"""
        SELECT COUNT(*)
        FROM {csv(filha)} f
        LEFT JOIN {csv(pai)} p ON f.{col_f} = p.{col_p}
        WHERE f.{col_f} IS NOT NULL AND p.{col_p} IS NULL
        """
    ).fetchone()[0]
    print(f"{filha}.{col_f} -> {pai}.{col_p}")
    print(f"    nulos={nulos:>6}  orfas={orfas:>6}")
 
print()
print("== Pedidos sem registros filhos ==")
for filha in [
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
]:
    sem_filhos = duckdb.sql(
        f"""
        SELECT COUNT(*)
        FROM {csv('olist_orders_dataset')} o
        LEFT JOIN (SELECT DISTINCT order_id FROM {csv(filha)}) f
               ON o.order_id = f.order_id
        WHERE f.order_id IS NULL
        """
    ).fetchone()[0]
    print(f"pedidos sem linhas em {filha}: {sem_filhos}")

== Registros orfaos (filha aponta para um pai que nao existe) ==
olist_orders_dataset.customer_id -> olist_customers_dataset.customer_id
    nulos=     0  orfas=     0
olist_order_items_dataset.order_id -> olist_orders_dataset.order_id
    nulos=     0  orfas=     0
olist_order_items_dataset.product_id -> olist_products_dataset.product_id
    nulos=     0  orfas=     0
olist_order_items_dataset.seller_id -> olist_sellers_dataset.seller_id
    nulos=     0  orfas=     0
olist_order_payments_dataset.order_id -> olist_orders_dataset.order_id
    nulos=     0  orfas=     0
olist_order_reviews_dataset.order_id -> olist_orders_dataset.order_id
    nulos=     0  orfas=     0
olist_products_dataset.product_category_name -> product_category_name_translation.product_category_name
    nulos=   610  orfas=    13

== Pedidos sem registros filhos ==
pedidos sem linhas em olist_order_items_dataset: 775
pedidos sem linhas em olist_order_payments_dataset: 1
pedidos sem linhas em olist_order_reviews_dat

In [11]:
import duckdb

duckdb.sql("""
SELECT DISTINCT p.product_category_name
FROM 'data/raw/olist_products_dataset.csv' p
LEFT JOIN 'data/raw/product_category_name_translation.csv' t
       ON p.product_category_name = t.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND t.product_category_name IS NULL
""").show()

┌───────────────────────────────────────────────┐
│             product_category_name             │
│                    varchar                    │
├───────────────────────────────────────────────┤
│ pc_gamer                                      │
│ portateis_cozinha_e_preparadores_de_alimentos │
└───────────────────────────────────────────────┘

